# Generate Scores

## Notebook Preferences

In [ ]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format="retina"

## Libraries

In [ ]:
import buckaroo  # noqa: F401
import lamindb as ln
import spatialdata as sd
import nbl
import itertools
import annsel as an

In [ ]:
ln.settings.sync_git_repo = "https://github.com/karadavis-lab/nbl.git"
ln.track("lQxax3ZyxtAm0001")

In [ ]:
ln.setup.settings.instance

## Load NBL Sdata

In [ ]:
nbl_sdata = sd.read_zarr(ln.Artifact.filter(key__contains="nbl.zarr", is_latest=True).one().path)

In [ ]:
nbl_wc = nbl_sdata.tables["wc_arcsinh"].an.filter(an.obs_col("pixie_cluster") == "NBL_cell")
nbl_sdata.update_annotated_regions_metadata(table=nbl_wc)

In [ ]:
nbl_sdata.tables["whole_cell"]

In [ ]:
nbl_sdata

In [ ]:
nbl_sdata.tables["nbl_wc"] = nbl_wc

nbl.util.write_elements(sdata=nbl_sdata, elements={"tables": "nbl_wc"})

In [ ]:
nbl_wc = nbl_sdata.tables["nbl_wc"]

### Get Marker Groups

## Compute Scores

In [ ]:
marker_groups = {
    "ADRN": nbl.ln.Adrenergic_Markers.to_list(),
    "ADRN_no_TH": nbl.ln.Adrenergic_Markers.to_list()[:2],
    "MESN": nbl.ln.Mesenchymal_Markers.to_list(),
}
nbl.tl.compute_marker_means(
    nbl_sdata, table_name="nbl_wc_diagnosis", layer=None, marker_groups=marker_groups, inplace=True
)

In [ ]:
for score_method, obs_1 in itertools.product(
    ["ratio", "normalized_difference", "log_ratio", "scaled_difference"], ["ADRN", "ADRN_no_TH"]
):
    nbl.tl.compute_score(
        sdata=nbl_sdata,
        table_name="nbl_wc_diagnosis",
        obs_1=f"{obs_1}_mean",
        obs_2="MESN_mean",
        score_method=score_method,
        score_col_name=f"{score_method}_no_TH" if "no_TH" in obs_1 else f"{score_method}",
        eps=1e-4,
    )

## Clustering with Area Normalized Arcsinh Transformed Markers

In [ ]:
ln.finish()